### 환경 설정

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('..')

import json
import glob
import numpy as np
import pandas as pd
from itertools import product
from sklearn.metrics import roc_auc_score
from src.train import tune_experiment, run_experiment

### LightGBM 파라미터 탐색

In [ ]:
# best_params_lgbm = tune_experiment(
#     name         = 'tuned',
#     use_features = True,
#     n_trials     = 50,
#     model_type   = 'lgbm',
# )

with open('../saved_models/tuned_best_params.json', encoding='utf-8') as f:
    best_params_lgbm = json.load(f)

### XGBoost 파라미터 탐색

In [ ]:
# best_params_xgb = tune_experiment(
#     name         = 'tuned_xgb',
#     use_features = True,
#     n_trials     = 50,
#     model_type   = 'xgb',
# )

with open('../saved_models/tuned_xgb_best_params.json', encoding='utf-8') as f:
    best_params_xgb = json.load(f)

### CatBoost 파라미터 탐색

In [ ]:
# best_params_cat = tune_experiment(
#     name         = 'tuned_catboost',
#     use_features = True,
#     n_trials     = 50,
#     model_type   = 'catboost',
# )

with open('../saved_models/tuned_catboost_best_params.json', encoding='utf-8') as f:
    best_params_cat = json.load(f)

### 튜닝 파라미터로 최종 학습

In [ ]:
result_lgbm = run_experiment(
    name         = 'exp08_tuned_lgbm',
    use_features = True,
    model_type   = 'lgbm',
    params       = best_params_lgbm,
)

result_xgb = run_experiment(
    name         = 'exp09_tuned_xgb',
    use_features = True,
    model_type   = 'xgb',
    params       = best_params_xgb,
)

result_cat = run_experiment(
    name         = 'exp10_tuned_cat',
    use_features = True,
    model_type   = 'catboost',
    params       = best_params_cat,
)

### 앙상블

In [ ]:
train = pd.read_csv('../data/raw/train.csv')
y = train['임신 성공 여부']

lgbm_oof = np.load('../saved_models/exp08_tuned_lgbm_oof.npy')
xgb_oof  = np.load('../saved_models/exp09_tuned_xgb_oof.npy')
cat_oof  = np.load('../saved_models/exp10_tuned_cat_oof.npy')

lgbm_test = np.load('../saved_models/exp08_tuned_lgbm_test.npy')
xgb_test  = np.load('../saved_models/exp09_tuned_xgb_test.npy')
cat_test  = np.load('../saved_models/exp10_tuned_cat_test.npy')

print(f'LightGBM OOF : {roc_auc_score(y, lgbm_oof):.4f}')
print(f'XGBoost  OOF : {roc_auc_score(y, xgb_oof):.4f}')
print(f'CatBoost OOF : {roc_auc_score(y, cat_oof):.4f}')

ensemble_oof = (lgbm_oof + xgb_oof + cat_oof) / 3
print(f'Ensemble OOF : {roc_auc_score(y, ensemble_oof):.4f}')

### 가중 앙상블

In [ ]:
best_score   = 0
best_weights = None

for w1, w2, w3 in product(range(0, 11), repeat=3):
    if w1 + w2 + w3 == 0:
        continue
    w1, w2, w3 = w1/10, w2/10, w3/10
    if abs(w1 + w2 + w3 - 1.0) > 1e-6:
        continue

    oof   = w1 * lgbm_oof + w2 * xgb_oof + w3 * cat_oof
    score = roc_auc_score(y, oof)

    if score > best_score:
        best_score   = score
        best_weights = (w1, w2, w3)

print(f'Best Weights  — LightGBM: {best_weights[0]}, XGBoost: {best_weights[1]}, CatBoost: {best_weights[2]}')
print(f'Best OOF ROC-AUC : {best_score:.4f}')

### 제출 파일 생성

In [ ]:
submission = pd.read_csv('../data/raw/sample_submission.csv')

ensemble_test_weighted = best_weights[0] * lgbm_test + best_weights[1] * xgb_test + best_weights[2] * cat_test
submission['probability'] = ensemble_test_weighted
submission.to_csv('../submissions/submission_ensemble_weighted_v1.csv', index=False)
print(submission.head())